# Notebook 5: Evaluation & Conclusions

**Goal:** Compile all experiment results, run ablation study, and draw conclusions.

This notebook produces the final comparison tables and charts for the poster.

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content'
REPO_DIR = f'{BASE}/{REPO}'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm')
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")

In [ ]:
# ============================================================
# DATA PATHS - auto-detected for Colab and Kaggle
# ============================================================
import os, glob as _glob

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'

if ENV == 'kaggle':
    # Look for processed data in Kaggle inputs (NB01 output added as dataset)
    _candidates = _glob.glob('/kaggle/input/*/processed/recipes.parquet')
    if _candidates:
        PROCESSED_DIR = os.path.dirname(_candidates[0])
        print(f"Found processed data: {PROCESSED_DIR}")
    else:
        # Same session as NB01, or data not yet added
        PROCESSED_DIR = '/kaggle/working/processed'
        print(f"Using: {PROCESSED_DIR}")
        print("If files are missing: run NB01 first, save a version, then add its output as a dataset here.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DIR = '/content/drive/MyDrive/food-recsys/processed'

print(f"PROCESSED_DIR = {PROCESSED_DIR}")

In [ ]:
with open(f'{PROCESSED_DIR}/results.json') as f:
    all_results = json.load(f)

print(f"Loaded results for {len(all_results)} configurations:")
for k in all_results:
    print(f"  {k}")

## 1. Grand Comparison Table

In [ ]:
CONFIG_LABELS = {
    'tfidf_svd':                    'TF-IDF + SVD(256)',
    'baseline_embedding':           'Pretrained (baseline text)',
    'no_recipe':                    '+ remove recipe_text',
    'no_recipe_no_macro':           '+ remove macro_tokens',
    'no_recipe_no_macro_with_ratios': '+ continuous ratios',
    'full_improved':                '+ extract ingredients',
    'behavioral_dish_vectors':      '+ behavioral dish vectors',
    'multi_vector_rrf':             '+ multi-vector (RRF)',
    'cross_encoder_rerank':         '+ cross-encoder rerank',
    'finetuned':                    'Fine-tuned model',
    'finetuned_full':               'Fine-tuned + all improvements',
}

metrics_to_show = ['P@5', 'P@10', 'R@20', 'NDCG@10', 'MRR']

rows = []
for cfg_key, label in CONFIG_LABELS.items():
    if cfg_key in all_results:
        row = {'Configuration': label}
        row.update({m: all_results[cfg_key].get(m, float('nan')) for m in metrics_to_show})
        rows.append(row)

grand_table = pd.DataFrame(rows).set_index('Configuration')
print("=== Grand Comparison Table ===")
print(grand_table.round(4).to_string())

## 2. Visualizations

In [ ]:
# Bar chart: P@10 across all configurations
plot_metric_comparison(
    {label: all_results[cfg] for cfg, label in CONFIG_LABELS.items() if cfg in all_results},
    metric_name='P@10',
    title='Precision@10 by Configuration',
)

In [ ]:
# Cumulative improvement line chart
cumulative_configs = [
    'baseline_embedding', 'no_recipe', 'no_recipe_no_macro',
    'no_recipe_no_macro_with_ratios', 'full_improved', 'behavioral_dish_vectors',
    'multi_vector_rrf', 'finetuned_full',
]
cumulative_labels = [
    'Baseline', '−recipe', '−macro_tok',
    '+ratios', '+ingredients', '+beh_vecs',
    '+multi_vec', '+fine_tune',
]
plot_cumulative_improvements(
    [all_results.get(c, {}) for c in cumulative_configs],
    metric_name='P@10',
    labels=cumulative_labels,
    title='Cumulative Improvement Steps (P@10)',
)

In [ ]:
# Heatmap: all metrics across all configurations
fig, ax = plt.subplots(figsize=(10, max(4, len(grand_table) * 0.5)))
sns.heatmap(
    grand_table[metrics_to_show].astype(float),
    annot=True, fmt='.3f', cmap='YlGn',
    linewidths=0.5, ax=ax,
)
ax.set_title('All Metrics by Configuration', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Ablation Study

Each improvement **in isolation** (not cumulative) — measures the independent contribution of each component.

In [ ]:
baseline_p10 = all_results.get('baseline_embedding', {}).get('P@10', 0)

ablation_configs = [
    ('no_recipe',           'Baseline − recipe_text only'),
    ('no_recipe_no_macro',  'Baseline − recipe − macro_tokens'),
    ('full_improved',       'Best text config (all improvements)'),
    ('behavioral_dish_vectors', 'Baseline + behavioral dish vecs only'),
    ('multi_vector_rrf',    'Best text + multi-vector only'),
    ('finetuned',           'Fine-tuned (best text config)'),
]

ablation_rows = []
for cfg, label in ablation_configs:
    if cfg in all_results:
        p10 = all_results[cfg].get('P@10', float('nan'))
        delta = p10 - baseline_p10
        ablation_rows.append({'Configuration': label, 'P@10': p10, 'Δ vs baseline': delta})

ablation_df = pd.DataFrame(ablation_rows).set_index('Configuration')
print("=== Ablation Study (each improvement in isolation) ===")
print(ablation_df.round(4).to_string())

## 4. Limitations & Discussion

### Limitations

**1. Proxy metric gap** — We evaluate on ratings (≥4 stars = positive), but our production system optimizes for orders. A 5-star rating may indicate aspiration while an order indicates concrete preference. The absolute metrics may not transfer, but **relative improvements (delta between configurations) are meaningful**.

**2. Missing user context** — Food.com lacks health profiles, allergies, nutritional goals, and temporal eating patterns. Our production embedding components (static profile, dynamic nutritional state) cannot be evaluated on this dataset. We focus on dish-side improvements.

**3. Domain mismatch** — Food.com is English-language, Western-centric home recipes. Our production catalog is Russian-language. Fine-tuning on Food.com provides food-domain understanding but may not capture culture-specific preferences.

**4. Survivorship bias** — Negative samples are unrated (not seen), not disliked. Hard negative mining partially addresses this but the noise is inherent.

### What we CAN confidently conclude
- Removing noisy text components (recipe instructions, discrete macro tokens) **consistently improves** embedding quality across all metrics
- **Dish-vector behavioral representation** outperforms text-based — the semantic gap between behavioral text and dish embeddings is real and measurable
- Domain fine-tuning **improves retrieval even with proxy labels**
- Multi-vector retrieval and cross-encoder reranking provide additional gains at the cost of latency

## 5. Production Recommendations

Based on our experiments:

**Immediate** (no additional data needed):
- Remove recipe text from dish embeddings
- Replace discrete macro tokens with continuous ratios
- Switch behavioral embedding to dish-vector aggregation with temporal decay

**Short-term** (requires interaction logging):
- Fine-tune embedding model on real order data (stronger signal than ratings)
- Implement multi-vector retrieval (separate taste, nutrition, context queries)

**Medium-term** (significant data volume):
- Cross-encoder reranking for top-N candidates
- Learned ranking model (LightGBM/XGBoost) to replace hand-tuned score formula
- Collaborative filtering (ALS/NCF) as additional retrieval channel

In [ ]:
# Export final results table to CSV for poster
grand_table.round(4).to_csv(f'{PROCESSED_DIR}/final_results.csv')
print("✓ Final results saved to data/processed/final_results.csv")
print("\nDone! Use final_results.csv and the charts above for the poster.")